In this workshop, we will gradually build an AI assistant:
1. Talk to a model
2. Create tools
3. Let the model use them

## Part 1: Connect to a model

To call a model from Python, we need three basic things:

1. **Install the required packages**
2. **Import the classes we want to use**
3. **Authenticate with an API key**

This is similar to using regular software on your computer, but not exactly the same.

For example:
- in a desktop app, you might install the software, open it, and sign in
- in Python, we install packages, import code from them, and provide credentials programmatically

So in this notebook, we will do the Python version of that process.

### Step 1: Install the required packages

**On a regular computer:** you might download an installer and run it.

**In Python:** we usually install packages with `pip`.

In this notebook we will install:

1. `langchain` – a framework that helps structure LLM-based applications
2. `langchain-openai` – the integration package for using OpenAI models through LangChain

In [3]:
!pip install -q langchain langchain-openai


[notice] A new release of pip available: 22.2.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Import the code we want to use

**In desktop software:** you open an application.

**In Python:** you import the classes or functions you want to use in your code.

An `import` statement does not “open the whole program.”  
It simply makes specific Python objects available in this notebook so we can use them.

In [ ]:
from langchain_openai import ChatOpenAI

### Step 3: Authenticate and create the model object

**In many apps:** you sign in with a username and password.

**When calling an API from Python:** you usually authenticate with an API key.

Now we will create a model object and store it in a variable called `llm`.

Here, `llm` refers to a language model interface we can call from code.

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key="WRITE THE KEY HERE")

This line creates a model object we can reuse to send prompts.

### Step 4: Send your first prompt to the model

Let's try the model.

Run the prompt:

`Introduce yourself. Which model are you?`

We are using `gpt-4o-mini`, which is often a good choice for lightweight demos.
Now we actually "talk" to the model.

In [ ]:
response = llm.invoke("Introduce yourself. Which model are you?")
response.pretty_print()

I am based on OpenAI's GPT-3 model, a sophisticated language model designed to understand and generate human-like text. My purpose is to assist with a variety of tasks, including providing information, answering questions, and engaging in conversation. How can I help you today?


#### What happened here?

- We sent a message to the model with `llm.invoke(...)`
- The model returned a response object
- `pretty_print()` displays that response in a readable way

At this stage, the model is only generating text.  
It does not have access to any external tools yet.

Try it yourself.  
Use the same code as before, but change only the prompt text.

In [ ]:
response = llm.invoke("Write one sentence about why tools are useful for LLMs.")
response.pretty_print()

================================== Ai Message ==================================

I am ChatGPT, based on the GPT-4 architecture developed by OpenAI. I'm designed to assist with a wide range of questions and tasks, providing information, answering queries, and engaging in conversation. How can I help you today?


Check point - is all good? Let's wait for the others...

## Part 2: Create a Python function

Before we give tools to the model, we need to create one.

In this step, we will create a simple Python function called `mojifuzzle`.

It will accept:
- an emoji name: `"heart"` or `"star"`
- a number that tells it how many emojis to generate

If it receives any other emoji name, it will return a puzzled emoji.

First, run the following cell to define the function.

Even though you are not a Python coder, and even though it is not required, try and read it and understand as much as possible. here are some epxlaination that might assist:

def - short for "define".
str - short for string, the technical term for textual information.
int – short for integer, which is the technical term for a whole number (i.e., not a decimal).
elif - short for else, if
* - means multipication, e.g. 2*3 = 6


In [ ]:
def mojifuzzle(emoji_name: str, count: int) -> str:
    if emoji_name == "heart":
        return "❤️" * count
    elif emoji_name == "star":
        return "⭐" * count
    else:
        return "🤔"

Now let's test the function with a few examples.
Run the next three cells:

In [ ]:
mojifuzzle("heart", 10)

'❤️❤️❤️❤️❤️❤️❤️❤️❤️❤️'

In [ ]:
mojifuzzle("star", 2)

'⭐⭐'

In [ ]:
mojifuzzle("moon", 2)

'🤔'

Notice that this function runs entirely in Python.  
No model is involved yet.

Check point - is all good? Let's wait for the others...

## Part 3: Turn the function into a tool schema

Suppose we ask the model:

`please mojifuzzle 3 times with a heart`

Run the next cell **again and again** and see what happens (and if the result is consistant).

In [ ]:
response = llm.invoke("Please mojifuzzle 3 times with a heart.")
response.pretty_print()

Sure! Here’s "mojifuzzle" with a heart emoji three times:

💖 mojifuzzle 💖 mojifuzzle 💖 mojifuzzle 💖


#### Observation

As you probably noticed, the model improvised.

It tried to respond based only on the text prompt and its own reasoning.  
It did **not** use our mojifuzzle.

Now let's make our function available to the model as a tool.

To do that, we will import the `tool` decorator and apply it to our function.

This does **not** mean the model executes the tool by itself yet.  
It means we are adding structured metadata so the model can understand:
- the tool's name
- its inputs
- its description

![Diagram](images/functionToTool.png)

In [ ]:
from langchain.tools import tool

@tool
def mojifuzzle(emoji_name: str, count: int) -> str:
    """Create a string of emojis using 'heart' or 'star'."""
    if emoji_name == "heart":
        return "❤️" * count
    elif emoji_name == "star":
        return "⭐" * count
    else:
        return "🤔"

The `@tool` decorator allows LangChain to extract a structured description of the function.

That structured description can then be shown to the model as a tool schema.

Here is one way to inspect that schema:

#### Optional: Run the following cell to see the schema created yourself:

In [7]:
from langchain_core.utils.function_calling import convert_to_openai_tool
import json

print(json.dumps(convert_to_openai_tool(mojifuzzle), indent=2))

{
  "type": "function",
  "function": {
    "name": "mojifuzzle",
    "description": "Create emojis using 'heart' or 'star'.",
    "parameters": {
      "properties": {
        "emoji_name": {
          "type": "string"
        },
        "count": {
          "type": "integer"
        }
      },
      "required": [
        "emoji_name",
        "count"
      ],
      "type": "object"
    }
  }
}


Check point - is all good? Let's wait for the others...

### Part 4: Bind the tool to the model

Now we will bind the tool to the model and store the result in a new variable called `llm_with_tools`.

In LangChain, `bind_tools(...)` tells the model which tool schemas it is allowed to use.

In [18]:
llm_with_tools = llm.bind_tools([mojifuzzle])

Now let's send the same request as before:

In [ ]:
response = llm_with_tools.invoke("Please mojifuzzle 3 times with a heart.")
response.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  mojifuzzle (call_ZTYX7DsmSyXhg4NUJLS90C4X)
 Call ID: call_ZTYX7DsmSyXhg4NUJLS90C4X
  Args:
    emoji_name: heart
    count: 3


We can also try a slightly more indirect request:

In [ ]:
response = llm_with_tools.invoke(
    "Please mojifuzzle the symbol of love for the number of sides in a triangle. Make sure to provide only a single output."
)
response.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  mojifuzzle (call_ZmfTwOGrEW2u6cKUzQz3my18)
 Call ID: call_ZmfTwOGrEW2u6cKUzQz3my18
  Args:
    emoji_name: heart
    count: 3


Let's examine the output.

This time, the model is no longer just improvising a plain-text answer.

Because the tool was bound to the model, the model can now return a **tool call request**.  
That means it can say, in structured form:

- call `mojifuzzle`
- with `emoji_name="heart"`
- and `count=3`

But the tool still has not been executed automatically in this notebook flow.

So at this point, the model can **request** a tool call, but something else must still:
1. read that request
2. execute the tool
3. send the tool result back to the model if needed

### Part 5: Close the loop and actually execute the tool

To actually use the tool, we need an orchestration loop.

At a high level, the loop works like this:

1. Send the user's message to the model
2. Check whether the model requested a tool call
3. If yes, execute the tool
4. Return the tool result to the model
5. Let the model produce the final answer

There are several ways to build this loop:

1. **Manually** – good for learning the mechanics
2. **With an agent abstraction** – convenient for many common cases
3. **With LangGraph** – useful when you want more explicit control over state, branching, retries, or multi-step workflows

So the important idea is not “agent vs graph” as two unrelated things.  
The important idea is that you need some orchestration layer to complete the tool-use cycle.

### Part 6: Expose the tool through an MCP server

To expose our tool through MCP, we can use an MCP server.

An MCP server does not “contain intelligence” by itself.  
Its role is to expose capabilities—such as tools, resources, and prompts—through a standard protocol that an MCP client can use.

In this example, the server exposes the `mojifuzzle` tool so that an MCP client (for example, Claude Desktop) can discover it and call it.

Here is the code:

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Mojifuzzle server")

@mcp.tool()
def mojifuzzle(emoji_name: str, count: int) -> str:
    """
    Create a string of emojis.
    Use 'heart' or 'star'.
    """
    if emoji_name == "heart":
        return "❤️" * count
    elif emoji_name == "star":
        return "⭐" * count
    else:
        return "🤔"

if __name__ == "__main__":
    mcp.run()

We would usually place this server code in a separate Python file and run it outside the notebook.

Then we can connect to it from an MCP client such as Claude Desktop.

In the live demo, the MCP client will:
1. discover the server
2. see that `mojifuzzle` is available as a tool
3. decide when to call it
4. receive the result and continue the conversation

### Key takeaway

- A **tool** is a callable capability
- An **agent or orchestration loop** is what decides when and how to use tools
- An **MCP server** is one way to expose tools to external AI clients through a standard protocol

Summarizing table of what we saw:

| Concept | What it is | Example here |
|---|---|---|
| Python function | Local code you can run directly | `mojifuzzle(...)` |
| Tool schema | Structured description the model can inspect | `@tool` + schema output |
| Tool call request | The model asking for a tool to be run | output from `llm_with_tools.invoke(...)` |
| Orchestrator / agent | The loop that executes the tool and continues the flow | LangChain agent / LangGraph / MCP client |
| MCP server | A server that exposes tools via a standard protocol | `FastMCP("Mojifuzzle server")` |